In [3]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing import sequence
import numpy as np

print("==================================================")
print("INICIANDO SISTEMA DE CLASIFICACIÓN DE EMOCIONES")
print("==================================================")

# 1. PREPROCESAMIENTO DE TEXTO
max_features = 10000  # Considerar solo las 10,000 palabras más frecuentes
maxlen = 200          # Longitud máxima fija para cada secuencia de texto

print("\n[Paso 1] Cargando y preprocesando el dataset de texto (IMDB)...")
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=max_features)

# Rellenar (padding) o truncar las secuencias para que todas tengan exactamente 'maxlen' tokens
x_train = sequence.pad_sequences(x_train, maxlen=maxlen)
x_test = sequence.pad_sequences(x_test, maxlen=maxlen)

print(f"Dimensiones de entrenamiento: {x_train.shape}")
print(f"Dimensiones de prueba: {x_test.shape}")


# 2. CONSTRUCCIÓN Y ENTRENAMIENTO DE LA RNN
print("\n[Paso 2] Construyendo la Red Neuronal Recurrente (RNN)...")

rnn_modelo = models.Sequential([
    layers.Embedding(input_dim=max_features, output_dim=128, input_length=maxlen),
    layers.LSTM(64, return_sequences=False), 
    layers.Dropout(0.2),                     
    layers.Dense(1, activation='sigmoid')    
])

# Compilar el modelo
rnn_modelo.compile(optimizer='adam', 
                  loss='binary_crossentropy', 
                  metrics=['accuracy'])

print(rnn_modelo.summary())

print("\nEntrenando la red neuronal ")
historia_rnn = rnn_modelo.fit(x_train, y_train, 
                              epochs=2, 
                              batch_size=64, 
                              validation_split=0.2)

# Evaluar el modelo con datos de prueba nuevos
loss, acc = rnn_modelo.evaluate(x_test, y_test, verbose=0)
print(f"\n¡Entrenamiento completado!")
print(f"-> Precisión del modelo en datos de prueba: {acc:.2%}")


# 3. SIMULACIÓN DE DESPLIEGUE E INTERFAZ DE PRUEBA
print("\n[Paso 3] Desplegando función de inferencia para la empresa...")

def predecir_emocion_feedback(comentario):
    """
    Función de inferencia que procesa un texto nuevo ingresado por un cliente,
    lo convierte al formato numérico y predice su sentimiento.
    """
    word_index = imdb.get_word_index()
    
    palabras = comentario.lower().split()
    secuencia = []
    for palabra in palabras:
        idx = word_index.get(palabra, 0)
        if idx < max_features:
            secuencia.append(idx)
        else:
            secuencia.append(2) 
            
    secuencia_padded = sequence.pad_sequences([secuencia], maxlen=maxlen)
    
    probabilidad = rnn_modelo.predict(secuencia_padded, verbose=0)[0][0]
    
    emocion = "Positivo 😊" if probabilidad >= 0.5 else "Negativo 😞"
    confianza = probabilidad if probabilidad >= 0.5 else (1 - probabilidad)
    
    return emocion, confianza

# Pruebas con ejemplos simulados de feedback de clientes
comentarios_prueba = [
    "El producto superó todas mis expectativas, la entrega fue rapidísima y funciona de maravilla.",
    "Pésimo servicio al cliente, el aparato llegó roto y nadie me ayudó a resolver el reembolso.",
    "Un artículo bastante regular, cumple con lo básico pero no destaca en nada."
]

print("\n--- PRUEBAS EN LA INTERFAZ DE CLIENTES ---")
for i, texto in enumerate(comentarios_prueba, 1):
    sentimiento, conf = predecir_emocion_feedback(texto)
    print(f"\nComentario {i}: '{texto}'")
    print(f"-> Emoción Detectada: {sentimiento} (Confianza: {conf:.1%})")

INICIANDO SISTEMA DE CLASIFICACIÓN DE EMOCIONES

[Paso 1] Cargando y preprocesando el dataset de texto (IMDB)...
Dimensiones de entrenamiento: (25000, 200)
Dimensiones de prueba: (25000, 200)

[Paso 2] Construyendo la Red Neuronal Recurrente (RNN)...


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

None

Entrenando la red neuronal 
Epoch 1/2
313/313 ━━━━━━━━━━━━━━━━━━━━ 40s 120ms/step - accuracy: 0.7752 - loss: 0.4688 - val_accuracy: 0.8246 - val_loss: 0.3894
Epoch 2/2
313/313 ━━━━━━━━━━━━━━━━━━━━ 40s 129ms/step - accuracy: 0.8925 - loss: 0.2703 - val_accuracy: 0.8718 - val_loss: 0.3440

¡Entrenamiento completado!
-> Precisión del modelo en datos de prueba: 85.54%

[Paso 3] Desplegando función de inferencia para la empresa...

--- PRUEBAS EN LA INTERFAZ DE CLIENTES ---

Comentario 1: 'El producto superó todas mis expectativas, la entrega fue rapidísima y funciona de maravilla.'
-> Emoción Detectada: Negativo 😞 (Confianza: 76.2%)

Comentario 2: 'Pésimo servicio al cliente, el aparato llegó roto y nadie me ayudó a resolver el reembolso.'
-> Emoción Detectada: Negativo 😞 (Confianza: 65.0%)

Comentario 3: 'Un artículo bastante regular, cumple con lo básico pero no destaca en nada.'
-> Emoción Detectada: Negativo 😞 (Confianza: 77.1%)
